In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import train_test_split

In [2]:
metadata = pd.read_csv('meta_bhak.csv', sep=';', index_col=0)

In [3]:
metadata

,Age (years),Gender,Condition
acc,,,
SRR9190548,36,Female,MDD
SRR9190515,27,Male,Healthy
SRR9190731,44,Female,SA
SRR9190772,51,Male,MDD
SRR9190614,23,Female,Healthy
...,...,...,...
SRR9190717,26,Female,Healthy
SRR9190489,40,Female,SA
SRR9190576,32,Male,SA


In [ ]:
df = pd.read_csv('./train_multi_omics_bhak/train_multi_omics_bhak.csv', sep=';', index_col=0)

In [ ]:
df

In [ ]:
X = df.T

In [ ]:
samples = X.index.intersection(metadata.index)

In [ ]:
X = X.loc[samples]
y = metadata.loc[samples, 'Age (years)']

In [ ]:
X = X.astype('float32')

In [ ]:
y

In [ ]:
X.isnull().sum()

In [ ]:
y.isnull().sum()

In [ ]:
def horvath_transform(age, adult_age=20):
    age = np.array(age)
    return np.where(age <= adult_age, np.log(age+1)-np.log(adult_age + 1), (age - adult_age)/(adult_age+1))

In [ ]:
def horvath_inverse(transformed_age, adult_age=20):
    transformed_aged = np.array(transformed_age)
    return np.where(transformed_age < 0, np.exp(transformed_age + np.log(adult_age+1))-1, transformed_age * (adult_age+1) + adult_age)

In [ ]:
y_transformed = horvath_transform(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)

In [ ]:
model = ElasticNetCV(l1_ratio=[.1, .5, .7, .9, .95, .99, 1],
                    cv=5,
                    n_jobs=4,
                    max_iter=10000)
model.fit(X_train, y_train)

In [ ]:
model.alpha

In [ ]:
model.l1_ratio_

In [ ]:
preds_transformed = model.predict(X_test)
preds_years = horvath_inverse(preds_transformed)
actual_years = horvath_inverse(y_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(actual_years, preds_years)
r2 = r2_score(actual_years, preds_years)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))
plt.scatter(actual_years, preds_years, alpha=0.6, color='teal')
plt.plot([0, 100], [0, 100], 'r--')
plt.title(f"Blood Clock: MAE = {mae:.2f} years (R² = {r2:.2f})")
plt.xlabel("Chronological Age")
plt.ylabel("Predicted DNAm Age")
plt.show()